# 01 - Khám phá dữ liệu flow
Mục tiêu: hiểu dữ liệu trước khi mô hình hoá (mất cân bằng lớp, chất lượng, đặc trưng phân biệt).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # de import duoc package `src`
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config
from src.preprocessing.data_loader import load_raw, load_splits, to_xy
cfg = load_config()

In [2]:
df = load_raw(cfg)
X, y, attack_type = to_xy(df, cfg)
print(df.shape)
attack_type.value_counts()

(389714, 78)


Label
Benign         380564
FTP-Patator      5931
SSH-Patator      3219
Name: count, dtype: int64

## Mất cân bằng lớp và bẫy accuracy

In [3]:
prev = y.mean()
print(f"Tỷ lệ tấn công: {prev:.3%}")
print(f"Accuracy của bộ phân loại 'luôn Benign': {1-prev:.4f}  (recall = 0)")

Tỷ lệ tấn công: 2.348%
Accuracy của bộ phân loại 'luôn Benign': 0.9765  (recall = 0)


## Chất lượng dữ liệu

In [4]:
print("NaN:", int(df.isna().sum().sum()), "| Inf:", int(np.isinf(X.to_numpy(dtype='float64')).sum()))
const = X.columns[X.nunique() <= 1].tolist()
print(len(const), "cột hằng số:", const)

NaN: 0 | Inf: 0
10 cột hằng số: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


## Đặc trưng khác biệt giữa Benign và Tấn công (trung vị, thang log)

In [5]:
g = X.groupby(np.where(y == 1, "Attack", "Benign")).median().T
g["ratio"] = (np.abs(g["Attack"]) + 1) / (np.abs(g["Benign"]) + 1)
g.assign(logratio=np.log10(g["ratio"])).sort_values("logratio", key=abs, ascending=False).head(15)

,Attack,Benign,ratio,logratio
Bwd IAT Total,9.113688e+06,3.000000,2.278422e+06,6.357634
Fwd IAT Std,9.638819e+05,0.000000,9.638829e+05,5.984024
Bwd IAT Std,8.420345e+05,0.000000,8.420355e+05,5.925330
Bwd IAT Max,2.567515e+06,3.000000,6.418790e+05,5.807453
Fwd IAT Total,6.117508e+06,47.000000,1.274481e+05,5.105333
Bwd IAT Mean,4.212638e+05,3.000000,1.053162e+05,5.022495
Fwd IAT Max,2.597869e+06,47.000000,5.412229e+04,4.733376
Init Fwd Win Bytes,2.920000e+04,0.000000,2.920100e+04,4.465398
Fwd IAT Mean,5.499312e+05,46.000000,1.170069e+04,4.068211
Flow Duration,9.114177e+06,48776.000000,1.868540e+02,2.271502


## Chia tập (chỉ số lưu ở `data/output/processed/splits.npz`)

In [6]:
sp = load_splits(df, cfg)
pd.DataFrame({k: {"n": len(v), "tấn công %": 100*y[v].mean()} for k, v in sp.items()}).T

,n,tấn công %
train,233828.0,2.34788
val,77943.0,2.34787
test,77943.0,2.34787
